## 7.0 Introduction

Lectures 4 and 5 built everything from vectors: data rows, centroids, principal components. The key operations were simple — add, scale, dot product, norm. Those are all *vector* operations.

This lecture introduces a new mathematical object — the **matrix** — and the operation that makes it powerful: **multiplication**. A matrix applied to a vector produces a new vector. That single idea turns out to be the engine behind an enormous range of applications: rotating shapes, transforming images, evolving populations through time, and computing the principal components that powered our PCA visualizations.

We will spend the first part of the lecture building matrix arithmetic from the ground up, with worked examples by hand and graphics to show exactly what is happening at each step. The second part applies everything to two concrete problems: rotating 2D shapes with a rotation matrix, and treating digital images as matrices.

By the end of this lecture you will be able to:

- Describe what a matrix is and identify its shape
- Perform matrix-matrix multiplication using the dot product method
- Perform scalar and Hadamard multiplication
- Interpret $A\mathbf{v}$ both as a sequence of dot products and as a linear combination of the columns of $A$
- Apply a rotation matrix to a vector or a set of points
- Load a grayscale image as a NumPy array and apply simple matrix transformations to it

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 7.1 What Is a Matrix?

A **matrix** is a rectangular array of numbers arranged in rows and columns. An $m \times n$ matrix has $m$ rows and $n$ columns:

$$A = \begin{bmatrix} a_{11} & a_{12} & \cdots & a_{1n} \\ a_{21} & a_{22} & \cdots & a_{2n} \\ \vdots & \vdots & \ddots & \vdots \\ a_{m1} & a_{m2} & \cdots & a_{mn} \end{bmatrix}$$

The entry in row $i$, column $j$ is written $a_{ij}$. In NumPy, a matrix is a two-dimensional array and `A.shape` returns `(m, n)`.

Everything we have done with vectors is a special case: a column vector $\mathbf{v} \in \mathbb{R}^n$ is an $n \times 1$ matrix; a row vector is $1 \times n$.

You will also encounter the word **tensor** in data science and machine learning. A tensor is a generalization of matrices to higher dimensions — a matrix is a 2D tensor, a 3D tensor is a cube of numbers (a color image, for example, has shape height $\times$ width $\times$ 3 for the RGB channels). For this course we work exclusively with 2D arrays, so matrix and 2D array are interchangeable.

The **transpose** of $A$, written $A^T$, converts rows to columns: $(A^T)_{ij} = a_{ji}$. If $A$ is $m \times n$ then $A^T$ is $n \times m$.

In [ ]:
A = np.array([[1, 2, 3],
              [4, 5, 6]])

print('A:')
print(A)
print('Shape:', A.shape)
print()
print('A transpose:')
print(A.T)
print('Shape:', A.T.shape)

**Question.** `A` is $2 \times 3$. What is the shape of `A.T`? If you transpose a column vector, what do you get?

## 7.2 Matrix-Matrix Multiplication

### The Shape Rule

Suppose $A$ is $m \times n$ and $B$ is $k \times p$. The product $AB$ is defined **only when $n = k$** — the inner dimensions must match. The result is $m \times p$.

$$\underbrace{A}_{m \times n} \underbrace{B}_{n \times p} = \underbrace{AB}_{m \times p}$$

This is worth checking every time before multiplying. If the inner dimensions don't match, the multiplication is undefined.

### Computing Entries: The Dot Product Method

The entry in row $i$, column $j$ of $AB$ is the **dot product of row $i$ of $A$ with column $j$ of $B$**:

$$(AB)_{ij} = \sum_{r=1}^{n} a_{ir} \, b_{rj}$$


In [ ]:
# By-hand multiplication diagram
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

A_ex = np.array([[1, 2, 3], [4, 5, 6]])
B_ex = np.array([[1, 0], [2, 1], [0, 2]])
AB_ex = A_ex @ B_ex

highlight_row = ['#d0e8f5', '#ffffff']   # steelblue tint for row 0, white for row 1
highlight_col = ['#ffd5cc', '#ffffff']   # tomato tint for col 0, white for col 1

def draw_matrix(ax, M, title, row_hl=None, col_hl=None, label=''):
    nrows, ncols = M.shape
    ax.set_xlim(-0.5, ncols - 0.5)
    ax.set_ylim(-0.5, nrows - 0.5)
    ax.set_aspect('equal')
    ax.invert_yaxis()
    ax.axis('off')
    ax.set_title(title, fontsize=11, pad=8)
    for i in range(nrows):
        for j in range(ncols):
            fc = 'white'
            if row_hl is not None and i == row_hl:
                fc = '#d0e8f5'
            if col_hl is not None and j == col_hl:
                fc = '#ffd5cc'
            if row_hl is not None and col_hl is not None and i == row_hl and j == col_hl:
                fc = '#d5f5d5'   # seagreen tint for intersection
            rect = plt.Rectangle((j - 0.48, i - 0.48), 0.96, 0.96,
                                  fc=fc, ec='#aaaaaa', lw=0.8)
            ax.add_patch(rect)
            ax.text(j, i, str(M[i, j]), ha='center', va='center', fontsize=12)
    # Bracket lines
    ax.plot([-0.5, -0.5], [-0.5, nrows - 0.5], 'k', lw=2)
    ax.plot([-0.5, -0.3], [-0.5, -0.5], 'k', lw=2)
    ax.plot([-0.5, -0.3], [nrows - 0.5, nrows - 0.5], 'k', lw=2)
    ax.plot([ncols - 0.5, ncols - 0.5], [-0.5, nrows - 0.5], 'k', lw=2)
    ax.plot([ncols - 0.5, ncols - 0.7], [-0.5, -0.5], 'k', lw=2)
    ax.plot([ncols - 0.5, ncols - 0.7], [nrows - 0.5, nrows - 0.5], 'k', lw=2)

# Panel 1: Highlight row 0 of A and col 0 of B -> entry (0,0)
draw_matrix(axes[0], A_ex, '$A$: row 0 highlighted', row_hl=0)
axes[0].text(3.2, 0.5, '·', ha='center', va='center', fontsize=20, color='black',
             transform=axes[0].transData)

# Panel 2: B with col 0 highlighted
ax2 = axes[1]
draw_matrix(ax2, B_ex, '$B$: col 0 highlighted', col_hl=0)

# Panel 3: Result AB
draw_matrix(axes[2], AB_ex, '$AB$: entry (0,0) = 5')
# Highlight entry (0,0)
rect = plt.Rectangle((-0.48, -0.48), 0.96, 0.96,
                     fc='#d5f5d5', ec='seagreen', lw=2)
axes[2].add_patch(rect)
axes[2].text(0, 0, '5', ha='center', va='center', fontsize=13, fontweight='bold')

plt.suptitle('Matrix Multiplication: Row $i$ of $A$ · Column $j$ of $B$ → Entry $(i,j)$ of $AB$',
             fontsize=11, y=1.02)
plt.tight_layout()
plt.show()


**Example.** Let

$$A = \begin{bmatrix} 1 & 2 & 3 \\ 4 & 5 & 6 \end{bmatrix}, \qquad B = \begin{bmatrix} 1 & 0 \\ 2 & 1 \\ 0 & 2 \end{bmatrix}$$

$A$ is $2 \times 3$ and $B$ is $3 \times 2$, so $AB$ is $2 \times 2$.

Entry $(0, 0)$: row 0 of $A$ dotted with column 0 of $B$:
$$[1, 2, 3] \cdot [1, 2, 0] = 1(1) + 2(2) + 3(0) = 1 + 4 + 0 = 5$$

Entry $(0, 1)$: row 0 of $A$ dotted with column 1 of $B$:
$$[1, 2, 3] \cdot [0, 1, 2] = 1(0) + 2(1) + 3(2) = 0 + 2 + 6 = 8$$

**Question.** Compute entries $(1, 0)$ and $(1, 1)$ by hand before running the code below.


In [ ]:
# Verify by-hand result with NumPy
A_ex = np.array([[1, 2, 3],
                 [4, 5, 6]])
B_ex = np.array([[1, 0],
                 [2, 1],
                 [0, 2]])

AB = A_ex @ B_ex
print('A @ B:')
print(AB)
print()
print('Shape of A:', A_ex.shape)
print('Shape of B:', B_ex.shape)
print('Shape of AB:', AB.shape)


**Question.** What is the shape of $BA$? Compute $BA$ and compare it to $AB$. Are they the same matrix? Does it even make sense to ask whether $AB = BA$ in general?

**Question.** Try `A_ex @ A_ex`. What error do you get and why?

In [ ]:
# BA has a different shape entirely
BA = B_ex @ A_ex
print('B @ A:')
print(BA)
print('Shape:', BA.shape)

## 7.3 Scalar and Hadamard Multiplication

Two other multiplication operations come up regularly and are worth distinguishing from matrix multiplication.

**Scalar multiplication** $\lambda A$ multiplies every entry of $A$ by the scalar $\lambda$. Shape is unchanged.

**Hadamard (elementwise) multiplication** $A \odot B$ multiplies corresponding entries. Both matrices must have the same shape. The result has the same shape. In NumPy: `A * B` or `np.multiply(A, B)`.

Hadamard multiplication is used in neural network layers and image masking, but it is *not* standard matrix multiplication. The notation $AB$ always means the dot-product version from Section 6.2.

In [ ]:
P = np.array([[1, 2],
              [3, 4]])
Q = np.array([[5, 6],
              [7, 8]])

print('Scalar multiplication (3 * P):')
print(3 * P)
print()
print('Hadamard multiplication (P * Q):')
print(P * Q)
print()
print('Matrix multiplication (P @ Q):')
print(P @ Q)

## 7.4 Matrix-Vector Multiplication

### Two Views of the Same Operation

Suppose $A$ is $m \times n$ and $\mathbf{v} \in \mathbb{R}^n$. The product $A\mathbf{v}$ is an $m$-dimensional vector. There are two equivalent ways to compute it, and each reveals something different.

---

**View 1 — Dot products (row view).** Each entry of $A\mathbf{v}$ is the dot product of a row of $A$ with $\mathbf{v}$:

$$(A\mathbf{v})_i = \mathbf{a}_i^T \cdot \mathbf{v} = \sum_{j=1}^n a_{ij} v_j$$

This is the mechanical way to compute the result entry by entry.

---

**View 2 — Linear combination (column view).** $A\mathbf{v}$ is a *weighted sum of the columns of $A$*, where the weights are the entries of $\mathbf{v}$:

$$A\mathbf{v} = v_1 \mathbf{a}_{1} + v_2 \mathbf{a}_{2} + \cdots + v_n \mathbf{a}_{n}$$

where $\mathbf{a}_{j}$ is the $j$-th column of $A$.

This view is the deeper one. It says that $A\mathbf{v}$ does not just multiply numbers — it **transforms** $\mathbf{v}$ into a new vector by combining the columns of $A$. The matrix $A$ defines a transformation; $\mathbf{v}$ tells us how much of each column to include.

---

**Example.** Let

$$A = \begin{bmatrix} 1 & 2 & 3 \\ 4 & 5 & 6 \end{bmatrix}, \qquad \mathbf{v} = \begin{bmatrix} 1 \\ -2 \\ 3 \end{bmatrix}$$

**Row view:**
$$(A\mathbf{v})_0 = 1(1) + 2(-2) + 3(3) = 1 - 4 + 9 = 6$$
$$(A\mathbf{v})_1 = 4(1) + 5(-2) + 6(3) = 4 - 10 + 18 = 12$$

**Column view:**
$$A\mathbf{v} = 1\begin{bmatrix}1\\4\end{bmatrix} + (-2)\begin{bmatrix}2\\5\end{bmatrix} + 3\begin{bmatrix}3\\6\end{bmatrix} = \begin{bmatrix}1\\4\end{bmatrix} + \begin{bmatrix}-4\\-10\end{bmatrix} + \begin{bmatrix}9\\18\end{bmatrix} = \begin{bmatrix}6\\12\end{bmatrix}$$

In [ ]:
# Diagram: row view vs column view of Av
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Matrix–Vector Multiplication: Two Equivalent Views', fontsize=12)

A_mv = np.array([[1, 2, 3], [4, 5, 6]])
v_mv = np.array([1, -2, 3])
result = A_mv @ v_mv   # [6, 12]

row_colors = ['steelblue', 'tomato']
col_colors = ['steelblue', 'tomato', 'seagreen']

# ── Left panel: row view ──────────────────────────────────────────────────────
# Show A as a grid of cells, v as a column, highlight each row·v dot product
ax = axes[0]
ax.set_title('Row view: each output entry = one dot product', fontsize=11)
ax.set_xlim(-0.3, 7.5)
ax.set_ylim(-0.3, 4.2)
ax.axis('off')

CELL = 0.8

def draw_cell(ax, x, y, val, fc='white', fontsize=11):
    rect = plt.Rectangle((x, y), CELL, CELL, fc=fc, ec='#aaaaaa', lw=0.7)
    ax.add_patch(rect)
    ax.text(x + CELL/2, y + CELL/2, str(val), ha='center', va='center', fontsize=fontsize)

# Draw A (2×3) with rows highlighted
ax.text(0.0, 3.7, 'A', fontsize=13, fontweight='bold', color='#333')
for r in range(2):
    for c in range(3):
        fc = row_colors[r] + '44' if True else 'white'
        # Use alpha via face color string trick — just use light tints
        tints = ['#d0e8f5', '#ffd5cc']
        draw_cell(ax, c * CELL, (1 - r) * CELL + 1.8, A_mv[r, c], fc=tints[r])

# Draw v (3×1) 
ax.text(2.75, 3.7, 'v', fontsize=13, fontweight='bold', color='#333')
for r in range(3):
    draw_cell(ax, 2.75, (2 - r) * CELL + 0.9, v_mv[r], fc='#f5f5f5')

# Equals sign
ax.text(3.75, 2.7, '=', fontsize=18, ha='center', va='center', color='#555')

# Draw result vector, each entry colored by its row
ax.text(4.3, 3.7, 'Av', fontsize=13, fontweight='bold', color='#333')
res_tints = ['#d0e8f5', '#ffd5cc']
for r in range(2):
    draw_cell(ax, 4.3, (1 - r) * CELL + 1.8, result[r], fc=res_tints[r])

# Annotations showing how each entry was computed
for r, color in enumerate(row_colors):
    y_text = 1.3 - r * 1.1
    row = A_mv[r]
    terms = ' + '.join([f'{row[j]}·({v_mv[j]:+d})' if v_mv[j] < 0
                        else f'{row[j]}·{v_mv[j]}' for j in range(3)])
    ax.text(0.0, y_text, f'Entry {r}: {terms} = {result[r]}',
            fontsize=9.5, color=color, fontweight='bold')

# ── Right panel: column view ──────────────────────────────────────────────────
# Show the three scaled columns as stacked bar-style vectors, then their sum
ax = axes[1]
ax.set_title('Column view: output = weighted sum of columns', fontsize=11)
ax.set_xlim(-0.5, 7.0)
ax.set_ylim(-12, 22)
ax.axis('off')

col_labels = [f'v₀·col 0\n= 1·[1, 4]', f'v₁·col 1\n= −2·[2, 5]', f'v₂·col 2\n= 3·[3, 6]']
scaled_cols = [v_mv[j] * A_mv[:, j] for j in range(3)]  # [1·[1,4], -2·[2,5], 3·[3,6]]
col_display = [[1,4], [-4,-10], [9,18]]

BAR_W = 1.0
gap = 1.6

for j, color in enumerate(col_colors):
    x = j * gap
    top_val, bot_val = col_display[j][0], col_display[j][1]

    # Draw two-element column vector as stacked cells
    ax.text(x + BAR_W/2, 20.5, col_labels[j], ha='center', va='center',
            fontsize=8.5, color=color, fontweight='bold', linespacing=1.5)

    for entry_i, val in enumerate([top_val, bot_val]):
        ypos = 12 - entry_i * 9
        height = abs(val) * 0.8 if val != 0 else 0.5
        ypos_bar = ypos - height if val >= 0 else ypos - height
        rect = plt.Rectangle((x, ypos - height), BAR_W, height,
                              fc=color, alpha=0.35, ec=color, lw=1.2)
        ax.add_patch(rect)
        ax.text(x + BAR_W/2, ypos - height/2, str(val),
                ha='center', va='center', fontsize=11, color=color, fontweight='bold')
        label = 'row 0' if entry_i == 0 else 'row 1'
        ax.text(x + BAR_W + 0.1, ypos - height/2, label,
                ha='left', va='center', fontsize=7.5, color='#888')

# Plus signs between columns
for j in range(2):
    x = j * gap + BAR_W + 0.3
    ax.text(x + 0.25, 7, '+', ha='center', va='center', fontsize=16, color='#555')

# Equals sign before result
ax.text(3 * gap - 0.3, 7, '=', ha='center', va='center', fontsize=16, color='#555')

# Draw result vector [6, 12]
x_res = 3 * gap + 0.1
ax.text(x_res + BAR_W/2, 20.5, 'Av\n= [6, 12]', ha='center', va='center',
        fontsize=9, color='black', fontweight='bold', linespacing=1.5)
for entry_i, val in enumerate([6, 12]):
    ypos = 12 - entry_i * 9
    height = val * 0.8
    rect = plt.Rectangle((x_res, ypos - height), BAR_W, height,
                          fc='#555', alpha=0.25, ec='#333', lw=1.5)
    ax.add_patch(rect)
    ax.text(x_res + BAR_W/2, ypos - height/2, str(val),
            ha='center', va='center', fontsize=11, color='#222', fontweight='bold')

plt.tight_layout()
plt.show()


**Question.** `A_mv` is $2 \times 3$ and `v_mv` has 3 entries. What would happen if `v_mv` had 2 entries instead? Try it and explain the error.

**Question.** In the column view, the weights on the columns of $A$ are the entries of $\mathbf{v}$. What does it mean geometrically when one of those weights is negative, as $v_1 = -2$ is here?

## 7.5 Application: Rotation Matrices

Matrix-vector multiplication transforms a vector into a new vector. One of the most elegant examples is the **2D rotation matrix**, which rotates a vector by angle $\theta$ counterclockwise around the origin:

$$R_\theta = \begin{bmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{bmatrix}$$

Applying $R_\theta$ to any vector $\mathbf{v} \in \mathbb{R}^2$ produces a vector of the same length, rotated by $\theta$. The matrix does not stretch or shrink — it only rotates.

**Example.** Rotate $\mathbf{v} = \begin{bmatrix}1\\0\end{bmatrix}$ by $45°$:

$$R_{45°} = \begin{bmatrix} \cos 45° & -\sin 45° \\ \sin 45° & \cos 45° \end{bmatrix} = \begin{bmatrix} \frac{\sqrt{2}}{2} & -\frac{\sqrt{2}}{2} \\ \frac{\sqrt{2}}{2} & \frac{\sqrt{2}}{2} \end{bmatrix} \approx \begin{bmatrix} 0.707 & -0.707 \\ 0.707 & 0.707 \end{bmatrix}$$

$$R_{45°}\begin{bmatrix}1\\0\end{bmatrix} = \begin{bmatrix}0.707\\0.707\end{bmatrix}$$

A unit vector pointing right gets rotated to point up and to the right at 45°. The length is preserved: $\sqrt{0.707^2 + 0.707^2} = 1$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Arc
from matplotlib.lines import Line2D

def rotation_matrix(deg):
    theta = np.radians(deg)
    return np.array([[np.cos(theta), -np.sin(theta)],
                     [np.sin(theta),  np.cos(theta)]])

v = np.array([1.0, 0.0])
angles = [45, 90, 135, 180, 270, 315]
t = np.linspace(0, 2 * np.pi, 300)

fig, axes = plt.subplots(2, 3, figsize=(13, 9))
axes = axes.flatten()

for ax, deg in zip(axes, angles):
    theta = np.radians(deg)
    R = rotation_matrix(deg)
    v_rot = R @ v
    cos_v, sin_v = np.cos(theta), np.sin(theta)

    # Unit circle and axes
    ax.plot(np.cos(t), np.sin(t), color='#cccccc', lw=1)
    ax.axhline(0, color='#cccccc', lw=0.7)
    ax.axvline(0, color='#cccccc', lw=0.7)

    # Original vector v (blue, solid)
    ax.annotate('', xy=v, xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color='#378ADD',
                                lw=2.2, mutation_scale=15))
    ax.plot(*v, 'o', color='#378ADD', ms=6, zorder=5)

    # Rotated vector Rv (orange, dashed)
    ax.annotate('', xy=v_rot, xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color='#D85A30',
                                lw=2.2, linestyle='dashed', mutation_scale=15))
    ax.plot(*v_rot, 'o', color='#D85A30', ms=6, zorder=5)

    # Rv label — offset so it doesn't overlap the arrowhead
    offset = np.array([0.08, 0.10])
    if v_rot[1] < -0.15:  offset = np.array([0.05, -0.18])
    if v_rot[0] < -0.4:   offset = np.array([-0.34, 0.06])
    ax.text(v_rot[0] + offset[0], v_rot[1] + offset[1],
            f'[{v_rot[0]:.2f}, {v_rot[1]:.2f}]',
            color='#D85A30', fontsize=9, fontweight='bold')

    # Arc showing the angle (always draw the short way round)
    if 2 < abs(deg) % 360 < 358:
        arc_r = 0.30
        sweep = deg % 360
        a1, a2 = (sweep - 360, 0) if sweep > 180 else (0, sweep)
        arc = Arc((0, 0), 2 * arc_r, 2 * arc_r,
                  angle=0, theta1=min(a1, a2), theta2=max(a1, a2),
                  color='#888888', lw=1.3)
        ax.add_patch(arc)
        mid = np.radians((a1 + a2) / 2)
        ax.text(arc_r * 1.75 * np.cos(mid), arc_r * 1.75 * np.sin(mid),
                f'{deg}°', fontsize=9.5, color='#555555',
                ha='center', va='center')

    ax.set_title(f'R({deg}°)  →  Rv = [{v_rot[0]:.2f}, {v_rot[1]:.2f}]',
                 fontsize=10, pad=5)

    # Matrix entries in bottom-left corner (monospaced, no LaTeX)
    mat_str = (f'[ {cos_v:+.2f}  {-sin_v:+.2f} ]\n'
               f'[ {sin_v:+.2f}  {cos_v:+.2f} ]')
    ax.text(-1.52, -1.25, mat_str, fontsize=8.5, color='#444444',
            va='bottom', fontfamily='monospace')
    ax.text(-1.52, -1.55, f'|Rv| = {np.linalg.norm(v_rot):.4f}',
            fontsize=8, color='#888888', style='italic')

    ax.set_xlim(-1.6, 1.6)
    ax.set_ylim(-1.6, 1.55)
    ax.set_aspect('equal')
    ax.tick_params(labelsize=8)

handles = [
    Line2D([0], [0], color='#378ADD', lw=2.5, label='v = [1, 0]  (original)'),
    Line2D([0], [0], color='#D85A30', lw=2.5, linestyle='dashed', label='Rv  (rotated)'),
]
fig.legend(handles=handles, loc='lower center', ncol=2,
           fontsize=11, frameon=False, bbox_to_anchor=(0.5, 0.005))

fig.suptitle('Rotation matrix R(\u03b8) applied to v = [1, 0]', fontsize=13, y=1.01)
plt.tight_layout(rect=[0, 0.06, 1, 1])
plt.show()


## 7.6 Application: Images as Matrices

A grayscale digital image is nothing more than a matrix of integers. Each entry is a **pixel** — a number between 0 (black) and 255 (white) representing brightness. An image with height $h$ and width $w$ is stored as an $h \times w$ matrix.

This means every image operation — flipping, rotating, brightening, blurring — is a matrix operation. The image *is* the matrix.

We will use a classic test image from the `scikit-image` library: a 512 × 512 photograph of a man with a camera, used by the image processing community as a standard benchmark for decades.

In [ ]:
from skimage import data

img = data.camera()   # 512 x 512 grayscale NumPy array, dtype uint8

print('Image type:  ', type(img))
print('Image shape: ', img.shape)
print('Dtype:       ', img.dtype)
print('Pixel range: ', img.min(), 'to', img.max())

In [ ]:
print(img)

In [ ]:
# Show the image and a small crop of raw pixel values
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.imshow(img, cmap='gray', vmin=0, vmax=255)
ax.set_title('Original image (512 × 512)')
ax.axis('off')

# Show pixel values in a 6x6 crop from the center
row_start, col_start = 250, 250
crop_size = 6
crop = img[row_start:row_start+crop_size, col_start:col_start+crop_size]

ax = axes[1]
ax.imshow(crop, cmap='gray', vmin=0, vmax=255)
for i in range(crop_size):
    for j in range(crop_size):
        ax.text(j, i, str(crop[i, j]), ha='center', va='center',
                fontsize=9, color='white' if crop[i, j] < 128 else 'black')
ax.set_title(f'Pixel values: {crop_size}×{crop_size} crop (rows {row_start}–{row_start+crop_size-1})')
ax.set_xticks([])
ax.set_yticks([])

plt.suptitle('A Grayscale Image Is a Matrix of Integers (0–255)', fontsize=12)
plt.tight_layout()
plt.show()

**Question.** The pixel values in the crop are all small numbers near 5. What does that tell you about the brightness of that region of the image? What part of the photograph do you think those pixels belong to?

**Question.** The image has shape `(512, 512)`. Row index 0 is the top of the image; row index 511 is the bottom. Column index 0 is the left; column index 511 is the right. Given this, what does `img[0, 0]` refer to in the photograph?

In [ ]:
# Matrix operations as image transformations
img_flip_h = np.fliplr(img)          # flip left-right: reverse column order
img_flip_v = np.flipud(img)          # flip top-bottom: reverse row order
img_rot90  = np.rot90(img)           # rotate 90° counterclockwise
img_bright = np.clip(img.astype(int) + 60, 0, 255).astype(np.uint8)   # brighten

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
titles = ['Horizontal flip (np.fliplr)', 'Vertical flip (np.flipud)', '90° rotation (np.rot90)', 'Brightened (+60) (np.clip)']
images = [img_flip_h, img_flip_v, img_rot90, img_bright]

for ax, image, title in zip(axes, images, titles):
    ax.imshow(image, cmap='gray', vmin=0, vmax=255)
    ax.set_title(title, fontsize=10)
    ax.axis('off')

plt.suptitle('Image Transformations as Matrix Operations', fontsize=12)
plt.tight_layout()
plt.show()

**Question.** `np.fliplr(img)` reverses the order of the columns. Write out what this operation does to the entry $a_{ij}$: what new index does it map to?

**Question.** The brightening operation adds 60 to every pixel value, then clips the result to the range $[0, 255]$. Why is clipping necessary? What would happen without it to pixels that were already bright (say, value 220)?

**Question.** A color image has shape $(h, w, 3)$ — the third dimension holds the red, green, and blue channels. How many numbers are stored in a $1920 \times 1080$ color image?

In [ ]:
# Show pixel value distributions before and after brightening
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.hist(img.flatten(), bins=50, color='steelblue', edgecolor='white', linewidth=0.3)
ax.set_xlabel('Pixel value')
ax.set_ylabel('Count')
ax.set_title('Original pixel distribution')
ax.set_xlim(0, 255)
ax.grid(True, linestyle='--', alpha=0.4)

ax = axes[1]
ax.hist(img_bright.flatten(), bins=50, color='tomato', edgecolor='white', linewidth=0.3)
ax.set_xlabel('Pixel value')
ax.set_ylabel('Count')
ax.set_title('Brightened pixel distribution (+60, clipped)')
ax.set_xlim(0, 255)
ax.grid(True, linestyle='--', alpha=0.4)

plt.suptitle('Effect of Brightness Transformation on Pixel Distribution', fontsize=12)
plt.tight_layout()
plt.show()

**Question.** The brightened histogram has a spike at 255. What does that spike represent, and why does it appear after clipping?



## 7.7 Repeated Matrix Multiplication and Iterated Maps

So far we have applied a matrix to a vector once. But what happens if we apply it *again and again*?

Define the sequence

$$\mathbf{x}_0, \quad \mathbf{x}_1 = A\mathbf{x}_0, \quad \mathbf{x}_2 = A\mathbf{x}_1 = A^2\mathbf{x}_0, \quad \mathbf{x}_3 = A^3\mathbf{x}_0, \quad \ldots$$

Each step multiplies the current state by $A$. The matrix $A$ is the *rule*; the vector $\mathbf{x}_t$ is the *state* at time $t$. This is called an **iterated map** or **linear dynamical system**.

This single idea powers an enormous range of models:

- **Population dynamics** — how many individuals are in each age group next year?
- **Epidemic spread** — how many people move from susceptible to infected to recovered each day?
- **PageRank** — how does a random web surfer's position distribute across pages over time?

We will explore a concrete example: a two-state population model.

### A Two-State Population Model

Imagine a population split into two regions, **Urban** (U) and **Rural** (R). Each year, some fraction of each group moves to the other:

- 90% of Urban residents stay Urban; 10% move to Rural.
- 80% of Rural residents stay Rural; 20% move to Urban.

We encode this as a **transition matrix** $A$, where entry $a_{ij}$ is the fraction of group $j$ that becomes group $i$ next year:

$$A = \begin{bmatrix} 0.9 & 0.2 \\ 0.1 & 0.8 \end{bmatrix}$$

The state vector $\mathbf{x}_t = \begin{bmatrix} U_t \\ R_t \end{bmatrix}$ holds the population fractions at time $t$, and $\mathbf{x}_{t+1} = A\mathbf{x}_t$.

**Note:** The columns of $A$ each sum to 1 — this ensures that total population is conserved at every step. A matrix with this property is called **column-stochastic**, and it is the defining structure of a Markov chain, which we will study in full next lecture.

**Question.** Check by hand: if $\mathbf{x}_0 = \begin{bmatrix}0.5\\0.5\end{bmatrix}$ (equal split), what is $\mathbf{x}_1 = A\mathbf{x}_0$? Does the total still sum to 1?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

A = np.array([[0.9, 0.2],
              [0.1, 0.8]])

x0 = np.array([0.5, 0.5])   # start: 50% Urban, 50% Rural

# Iterate: apply A repeatedly
n_steps = 30
states = [x0]
x = x0.copy()
for _ in range(n_steps):
    x = A @ x
    states.append(x.copy())

states = np.array(states)   # shape (31, 2)

print('Step 0:', states[0])
print('Step 1:', states[1])
print('Step 2:', states[2])
print('Step 5:', states[5])
print('Step 30:', states[30])
print()
print('Column sums of A (should both be 1.0):', A.sum(axis=0))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# ── Left: trajectory of state vector over time ────────────────────────────
ax = axes[0]
ax.plot(states[:, 0], color='steelblue', lw=2, marker='o', markersize=3, label='Urban fraction')
ax.plot(states[:, 1], color='tomato',    lw=2, marker='o', markersize=3, label='Rural fraction')
ax.axhline(2/3, color='steelblue', lw=1, linestyle='--', alpha=0.5, label='Urban steady state (2/3)')
ax.axhline(1/3, color='tomato',    lw=1, linestyle='--', alpha=0.5, label='Rural steady state (1/3)')
ax.set_xlabel('Time step')
ax.set_ylabel('Population fraction')
ax.set_title('State vector converges to a fixed point')
ax.legend(fontsize=9)
ax.set_ylim(0, 1)
ax.grid(True, linestyle='--', alpha=0.4)

# ── Right: starting vectors in 2D state space, then trajectories ──────────
ax = axes[1]
starts = [
    np.array([1.0, 0.0]),   # all Urban
    np.array([0.0, 1.0]),   # all Rural
    np.array([0.3, 0.7]),
    np.array([0.8, 0.2]),
]
colors = ['steelblue', 'tomato', 'seagreen', 'darkorchid']
labels = ['Start [1.0, 0.0]', 'Start [0.0, 1.0]', 'Start [0.3, 0.7]', 'Start [0.8, 0.2]']

for x0_i, color, label in zip(starts, colors, labels):
    # Draw the starting vector as an arrow from the origin
    ax.annotate('', xy=x0_i, xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color=color, lw=2.0, mutation_scale=13))
    # Compute and plot the trajectory in 2D state space
    traj = [x0_i.copy()]
    xi = x0_i.copy()
    for _ in range(20):
        xi = A @ xi
        traj.append(xi.copy())
    traj = np.array(traj)
    ax.plot(traj[:, 0], traj[:, 1], color=color, lw=1.2, alpha=0.5,
            marker='o', markersize=2.5, label=label)

# Mark the steady state
steady = np.array([2/3, 1/3])
# Draw the steady state as a black vector from the origin
ax.annotate('', xy=steady, xytext=(0, 0),
            arrowprops=dict(arrowstyle='->', color='black', lw=2.5, mutation_scale=15))
ax.plot([], [], color='black', lw=2.5, label='Steady state (2/3, 1/3)')

ax.set_xlabel('Urban fraction')
ax.set_ylabel('Rural fraction')
ax.set_title('State-space view: starting vectors and trajectories')
ax.set_xlim(-0.05, 1.1)
ax.set_ylim(-0.05, 1.1)
ax.set_aspect('equal')
ax.legend(fontsize=8)
ax.grid(True, linestyle='--', alpha=0.4)

plt.suptitle('Iterated Map: $\mathbf{x}_{t+1} = A\mathbf{x}_t$  (Two-State Population Model)', fontsize=12)
plt.tight_layout()
plt.show()

**Question.** No matter where you start, the system converges to Urban ≈ 2/3, Rural ≈ 1/3. Call this the **steady state** $\mathbf{x}^*$. Verify algebraically that $A\mathbf{x}^* = \mathbf{x}^*$ when $\mathbf{x}^* = \begin{bmatrix}2/3 \\ 1/3\end{bmatrix}$.

**Question.** The equation $A\mathbf{x}^* = \mathbf{x}^*$ can be rewritten as $A\mathbf{x}^* = 1 \cdot \mathbf{x}^*$. In words: applying $A$ to $\mathbf{x}^*$ *scales* it by exactly 1 — it doesn't change direction at all. Keep this in mind. It is not a coincidence.

**Question.** What do you think would happen if we iterated $A$ 1000 times? Try computing `np.linalg.matrix_power(A, 1000) @ np.array([1.0, 0.0])` and inspect the result.